# PY300 — Box Plots and Outlier Detection

An **outlier** is a data point that is significantly different from the rest. Outliers can be genuine (a CEO's salary in employee data) or errors (a typo turning 50 into 5000). Detecting them is a critical EDA step.

## Topics Covered
1. Box plot anatomy and interpretation
2. Outlier detection using IQR (box plot method)
3. Outlier detection using Z-Score
4. Outlier detection using Modified Z-Score (MAD)
5. Outlier detection using Percentile Capping

> **Perspective:** There is no single "correct" way to detect outliers. The right method depends on your data distribution and domain knowledge. A heart rate of 200 bpm is always an outlier; a salary of $500K might be perfectly normal in Silicon Valley.

---
## 1. The Box Plot — Anatomy

A box plot (box-and-whisker plot) visualizes the **five-number summary**:

```
         ┌─────────────┐
    ─────┤      |      ├───── o    o
         └─────────────┘
    Min  Q1    Q2     Q3  Max  Outliers
    (whisker)  (box)  (whisker)
```

| Component | What It Shows |
|---|---|
| **Box** | Middle 50% of data (Q1 to Q3 = IQR) |
| **Line inside box** | Median (Q2) |
| **Whiskers** | Extend to the farthest point within 1.5 × IQR from the box |
| **Dots beyond whiskers** | Outliers |

In [ ]:
## Creating a box plot with Matplotlib

import matplotlib.pyplot as plt
import numpy as np

# Sample data with outliers
np.random.seed(42)
normal_data = np.random.normal(loc=50, scale=10, size=100)
outliers = np.array([5, 8, 95, 100, 110])  # Intentional outliers
data = np.concatenate([normal_data, outliers])

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Horizontal box plot
axes[0].boxplot(data, vert=False)
axes[0].set_title('Box Plot — Horizontal')
axes[0].set_xlabel('Value')

# Vertical box plot with notch (shows confidence interval around median)
axes[1].boxplot(data, notch=True, patch_artist=True,
                boxprops=dict(facecolor='lightblue'))
axes[1].set_title('Box Plot — Notched (Median CI)')
axes[1].set_ylabel('Value')

plt.tight_layout()
plt.show()

---
## 2. Outlier Detection: IQR Method (Box Plot Rule)

The most common method. A point is an outlier if it falls outside:
- **Lower bound:** Q1 − 1.5 × IQR
- **Upper bound:** Q3 + 1.5 × IQR

**When to use:** Works well for approximately symmetric distributions. Simple and widely understood.

In [ ]:
## IQR-based outlier detection

import pandas as pd
import numpy as np

np.random.seed(42)
data = np.concatenate([np.random.normal(50, 10, 100), [5, 8, 95, 100, 110]])
df = pd.DataFrame({'value': data})

Q1 = df['value'].quantile(0.25)
Q3 = df['value'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = df[(df['value'] < lower_bound) | (df['value'] > upper_bound)]

print(f"Q1 = {Q1:.2f}, Q3 = {Q3:.2f}, IQR = {IQR:.2f}")
print(f"Lower bound: {lower_bound:.2f}")
print(f"Upper bound: {upper_bound:.2f}")
print(f"\nOutliers found: {len(outliers)}")
print(outliers.sort_values('value'))

---
## 3. Outlier Detection: Z-Score Method

The **Z-Score** measures how many standard deviations a point is from the mean.

```
Z = (x - mean) / std
```

Typically, a point with |Z| > 3 is considered an outlier (i.e., more than 3 standard deviations from the mean).

**When to use:** Data is roughly normally distributed. Sensitive to extreme outliers (because mean and std are themselves affected by outliers).

In [ ]:
## Z-Score outlier detection

from scipy import stats

z_scores = np.abs(stats.zscore(df['value']))
threshold = 3

z_outliers = df[z_scores > threshold]

print(f"Z-Score threshold: |Z| > {threshold}")
print(f"Outliers found: {len(z_outliers)}")
print(z_outliers)

# Learning Point: Z-Score relies on mean and std, which are themselves
# distorted by outliers. This is a chicken-and-egg problem — the very
# outliers you want to detect are affecting the detection threshold.
# The Modified Z-Score (next) fixes this.

---
## 4. Outlier Detection: Modified Z-Score (MAD)

Uses the **Median Absolute Deviation** (MAD) instead of mean/std. MAD is robust to outliers because the median is not distorted by extreme values.

```
Modified Z = 0.6745 × (x - median) / MAD
```

**When to use:** Preferred over Z-Score when you suspect many outliers or skewed data.

In [ ]:
## Modified Z-Score using Median Absolute Deviation (MAD)

median = df['value'].median()
mad = np.median(np.abs(df['value'] - median))
modified_z = 0.6745 * (df['value'] - median) / mad

threshold = 3.5    # Common threshold for modified Z-Score
mad_outliers = df[np.abs(modified_z) > threshold]

print(f"Median = {median:.2f}, MAD = {mad:.2f}")
print(f"Modified Z-Score threshold: |MZ| > {threshold}")
print(f"Outliers found: {len(mad_outliers)}")
print(mad_outliers.sort_values('value'))

# Best Practice: The Modified Z-Score is often the most reliable default
# choice because it is robust to the very outliers it is trying to detect.

---
## 5. Outlier Detection: Percentile Capping (Winsorization)

Cap values below the 1st or 5th percentile and above the 95th or 99th percentile. Simple and domain-agnostic.

**When to use:** Quick and effective for mild outlier handling. Commonly used in financial and marketing data.

In [ ]:
## Percentile-based outlier detection

p5 = df['value'].quantile(0.05)
p95 = df['value'].quantile(0.95)

pct_outliers = df[(df['value'] < p5) | (df['value'] > p95)]

print(f"5th percentile : {p5:.2f}")
print(f"95th percentile: {p95:.2f}")
print(f"Outliers (outside 5th-95th): {len(pct_outliers)}")
print(pct_outliers.sort_values('value'))

---
## Comparison: Which Method When?

| Method | Based On | Robust to Outliers? | Best For |
|---|---|---|---|
| **IQR (Box Plot)** | Q1, Q3, IQR | Yes (uses quartiles) | General purpose, visual reporting |
| **Z-Score** | Mean, Std | No (mean/std shift) | Normally distributed data |
| **Modified Z-Score (MAD)** | Median, MAD | Yes | Skewed data, many outliers |
| **Percentile Capping** | Arbitrary percentiles | Yes | Quick cleaning, business rules |
| **Isolation Forest** (ML) | Tree splits | Yes | High-dimensional data (advanced) |
| **DBSCAN** (ML) | Density | Yes | Spatial/cluster-based outliers (advanced) |

> **Key Takeaway:** Detection is step one. What you *do* with outliers (remove, cap, transform, or keep) depends on domain knowledge. An outlier in medical data might be a life-saving signal. An outlier in survey data might be a typo. Context is everything.